# Lab: recording Ground Station handovers
A "handover" happens when a Ground Station's active link to a satellite drops and (possibly) another satellite takes over. In this lab you will:
1. build **one constellation** (a single Walker Delta shell) and **one Ground Station**, connected with the `best-angle-until-disconnection` strategy: stay on the same satellite as long as it is visible, then switch to the best newly visible one
2. register a `record_handover` action on the `time_manager`, exactly like `record_rx_state` in [sat_com_topology_c4.ipynb](sat_com_topology_v4_networkx.ipynb) or `compute_shortest_path` in [shortest_path_lab.ipynb](shortest_path_lab.ipynb)
3. run the simulation and, every time the Ground Station's link disconnects from its current satellite, record the timestep at which it happened

In [ ]:
from sat_com_builder.configuration_manager import BaseConfigurationManager
from sat_com_builder.models import SimulationProperty


## 1. One constellation, one Ground Station
The Ground Station is loaded from [`configurations/single_ground_station.txt`](configurations/single_ground_station.txt), which contains a single row (Toulouse). Its `connectivity_properties` use the `best-angle-until-disconnection` strategy, which is exactly what produces handovers: the Ground Station keeps a single active link and only switches satellite once the current one drops below the horizon.

We simulate 30 minutes, ticking every 15 seconds, to see a few handovers happen.

In [ ]:
lab_config = {
    "simulation_name": "Handover Lab",
    "start_date": "2026-01-01 00:00:00.000000",
    "end_date": "2026-01-01 00:30:00.000000",
    "movement_model": "pyorbital",
    "distance_model": "sklearn",
    "ground_objects_properties": [
        {
            "identifier": "Single Ground Station",
            "data_file": "./configurations/single_ground_station.txt",
            "type": "ground_station",
            "connectivity_properties": {
                "elevation_above_horizon": 20,
                "ground_to_space_connections_strategy": "best-angle-until-disconnection",
            },
        },
    ],
    "walker_shells": [
        {
            "type": "delta",
            "constellation_property": {
                "identifier": "Handover Lab Walker",
                "amount_of_orbit_plane": 8,
                "amount_of_satellite_per_orbit_plane": 10,
                "inclination": 53.0,
                "phase_difference_between_satellites": True,
                "mean_revolution_per_day": 15.05,
            },
            "orbital_connectivity_property": {
                "adjacent_inter_satellite_shifting": 0,
                "maximum_inter_satellite_count": 4,
                "maximum_inter_satellite_range_distance": 6000,
                "maximum_ground_station_range": 2000,
                "maximum_user_terminal_range": 2000,
                "maximum_connected_ground_object": 10000,
                "maximum_connected_user_terminal": 1000,
                "maximum_connected_ground_station": 10,
            },
            "ground_object_white_list": [],
        }
    ],
}

simulation_properties = SimulationProperty(**lab_config)
configuration_manager = BaseConfigurationManager(simulation_property=simulation_properties)

simulation_manager = configuration_manager.load_simulation()

ground_station = simulation_manager.get_ground_stations()[0]

print(f"{len(simulation_manager.get_satellites())} satellites loaded")
print(f"Ground Station: {ground_station.label} (object_id={ground_station.object_id})")


## 2. Record every handover
At every tick, the default update protocol (registered by `set_time_manager` during `load_simulation`) recomputes the Ground Station's link according to its strategy: it removes the link if the current satellite is no longer visible, and connects to a new one if a better one is available.

Our `record_handover` action runs right after that, on every tick, and compares the currently connected satellite to the one connected on the previous tick:
- **no previous satellite** → this is the first connection, not a handover
- **same satellite as before** → nothing happened
- **different satellite (including no satellite at all)** → the link disconnected, we record the timestep

> **Note:** `SimulationManager.get_connected_satellites_to_ground_station` in `sat_com_topology==4.2.0` collects results in a `set()`, but `Satellite` overrides `__eq__` without `__hash__`, so Python makes it unhashable — this raises `TypeError: unhashable type: 'Satellite'`. Until this is fixed upstream, we use `get_ground_station_links_connected_to_a_ground_station` instead, which returns a plain `list` and works around the issue.

In [ ]:
def get_connected_satellite_id(ground_station):
    connected_links = simulation_manager.get_ground_station_links_connected_to_a_ground_station(
        ground_station.object_id
    )
    if not connected_links:
        return None

    link = connected_links[0]
    connected_satellite = link.destination if link.source == ground_station else link.source
    return connected_satellite.topology_uniq_id


handover_events = []
connection_state = {"current_satellite_id": None}


def record_handover():
    new_satellite_id = get_connected_satellite_id(ground_station)

    previous_satellite_id = connection_state["current_satellite_id"]
    link_disconnected = (
        previous_satellite_id is not None and new_satellite_id != previous_satellite_id
    )

    if link_disconnected:
        current_time = simulation_manager.time_manager.current_time

        handover_events.append(
            {
                "time": current_time,
                "previous_satellite": previous_satellite_id,
                "new_satellite": new_satellite_id,
            }
        )

        if new_satellite_id is not None:
            print(
                f"[{current_time:%H:%M:%S}] handover: Satellite#{previous_satellite_id} -> Satellite#{new_satellite_id}"
            )
        else:
            print(
                f"[{current_time:%H:%M:%S}] disconnected from Satellite#{previous_satellite_id}, no satellite in view"
            )

    connection_state["current_satellite_id"] = new_satellite_id


simulation_manager.time_manager.register_action(record_handover)


## 3. Run the simulation

In [ ]:
simulation_manager.time_manager.tick_until_the_end(ticking_time_in_seconds=15)


## 4. Recap
The recorded timestep of each handover is available in `handover_events`.

In [ ]:
handover_timesteps = [event["time"] for event in handover_events]


def describe_handover(event):
    new_satellite = (
        f"Satellite#{event['new_satellite']}" if event["new_satellite"] is not None else "no satellite in view"
    )
    return f"- {event['time']:%H:%M:%S} : Satellite#{event['previous_satellite']} -> {new_satellite}"


print(f"{len(handover_events)} handover(s) recorded")
for event in handover_events:
    print(describe_handover(event))


## What's next?
- Lower `elevation_above_horizon` to keep satellites connected longer (fewer, later handovers) or raise it for more frequent ones.
- Swap `best-angle-until-disconnection` for `everything-visible` and see why the notion of "handover" stops making sense (the Ground Station connects to every visible satellite at once instead of picking one).
- Combine this with [shortest_path_lab.ipynb](shortest_path_lab.ipynb): record a handover event on the User Terminal too, and check whether both sides ever hand over to the same satellite at the same time.